# 09 — Explainability: SHAP, PDP, Permutation Importance

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))
import numpy as np, joblib, shap, matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from src.data import load_processed
from src.config import MODELS_DIR, FIGURES_DIR, FEATURE_COLS

In [ ]:
X_val, y_val = load_processed('val')
model = joblib.load(MODELS_DIR / 'XGBoost_tuned.joblib') if (MODELS_DIR / 'XGBoost_tuned.joblib').exists() else joblib.load(MODELS_DIR / 'XGBoost.joblib')

## SHAP TreeExplainer

In [ ]:
sample = X_val[:5000]
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(sample)
shap.summary_plot(shap_values, sample, feature_names=FEATURE_COLS, show=False)
plt.tight_layout(); plt.savefig(FIGURES_DIR / '09_shap_summary.png', dpi=120, bbox_inches='tight'); plt.show()

## Permutation Importance

In [ ]:
pi = permutation_importance(model, X_val[:20_000], y_val[:20_000], n_repeats=5, random_state=42, n_jobs=-1)
import pandas as pd
pd.Series(pi.importances_mean, index=FEATURE_COLS).sort_values().plot.barh(figsize=(7,4), color='darkorange')
plt.title('Permutation Importance'); plt.tight_layout()
plt.savefig(FIGURES_DIR / '09_perm_importance.png', dpi=120); plt.show()

## Partial Dependence Plots

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
PartialDependenceDisplay.from_estimator(model, X_val[:30_000], features=list(range(len(FEATURE_COLS))), feature_names=FEATURE_COLS, ax=ax)
plt.tight_layout(); plt.savefig(FIGURES_DIR / '09_pdp.png', dpi=120); plt.show()